# GLAAM Address Matching Notebook

Matches LDC (London Datastore) messy addresses against OS NGD reference data
to find UPRN links.

## Strategy

Neither the LDC nor the OS data has perfect, consistently-structured address
components. To maximise recall we run **six matching rounds**, each representing
a different way of encoding addresses into a single string:

| Round | Canonical key (OS)                         | Messy key (LDC)              |
|-------|--------------------------------------------|------------------------------|
| **a** | org name + street + postcode sector        | org name + street + sector   |
| **b** | fields 1-4 + street + sector               | full address tokens + sector |
| **c** | building name + number + street + sector   | ↑ same messy b               |
| **d** | building name + subname + street + sector  | ↑ same messy b               |
| **e** | number + street + sector                   | ↑ same messy b               |
| **f** | name + subname + number + street + sector  | ↑ same messy b               |

For each round, `AddressMatcher` runs an exact-match pass followed by a
Splink probabilistic pass. Results are unioned and **hit rules** are applied
to decide which matches to accept.

## Hit rules (post-matching)

A premises is considered **matched** if any of these conditions hold:
1. `fuzz_similarity == 100`  — the input and reference address strings are identical
2. `fuzz_similarity > 95` **and** `method == 'a'` — org-name method, near-exact
3. `distinguishability IS NOT NULL` — Splink found a uniquely distinguishing token

This notebook produced **~77% coverage** on the March 2026 LDC dataset.

## Prerequisites

Data files required (place in `../data/` relative to this notebook):
- `../data/ngd/` — OS NGD ZIP files (downloaded from the run-mar-2026 folder)
- `../data/ldc/ldc_history_YYYY-MM-DD.csv` — set `LDC_FILE` below

In [ ]:
# ============================================================
#  CONFIGURATION — edit these paths before running
# ============================================================
import os

NGD_DATA_DIR = os.path.join("..", "data", "ngd")   # folder containing OS NGD ZIPs
LDC_FILE     = os.path.join("..", "data", "ldc", "ldc_history_2025-12-31.csv")

## 1. Imports and DuckDB setup

In [ ]:
import multiprocessing
import os
import tempfile
import time
import zipfile

import duckdb
import numpy as np
import pandas as pd
import psutil
from thefuzz import fuzz

from uk_address_matcher import (
    AddressMatcher,
    ExactMatchStage,
    SplinkStage,
)

# -----------------------------------------------------------------
# DuckDB connection — configured once, never replaced
# -----------------------------------------------------------------
con = duckdb.connect(database=":memory:")

cores = multiprocessing.cpu_count()
con.execute(f"PRAGMA threads={cores}")

# Respect DUCKDB_MEMORY_LIMIT env var (set by Docker); else use 90% RAM
container_memory = os.environ.get("DUCKDB_MEMORY_LIMIT", "")
if container_memory:
    con.execute(f"PRAGMA memory_limit='{container_memory}'")
    mem_label = container_memory
else:
    total_gb  = psutil.virtual_memory().total // (1024 ** 3)
    mem_limit = int(total_gb * 0.9)
    con.execute(f"PRAGMA memory_limit='{mem_limit}GB'")
    mem_label = f"{mem_limit}GB (auto)"

temp_dir = tempfile.gettempdir()
con.execute(f"SET temp_directory='{temp_dir}'")

print(f"DuckDB ready — threads: {cores}, memory: {mem_label}, temp: {temp_dir}")

## 2. Helper functions

In [ ]:
# Column sets expected in OS NGD product files
_DEFAULT_COLS = [
    "uprn", "organisationname", "subname", "name", "number",
    "streetname", "locality", "townname",
    "primaryclassificationdescription", "postcode",
    "fulladdress", "latitude", "longitude",
]
_ALT_COLS = [
    "uprn", "alternatesubname", "alternatename", "alternatenumber",
    "streetname", "locality", "townname", "postcode",
    "fulladdress", "addressstatus",
]
_PSTL_COLS = [
    "uprn", "organisationname", "subbuildingname", "buildingname",
    "buildingnumber", "thoroughfare", "dependentlocality",
    "posttown", "postcode",
]


def _number_str(series: pd.Series) -> pd.Series:
    """Convert nullable integer series to strings, replacing <NA> with empty."""
    return series.astype(str).replace("<NA>", "").replace("nan", "")


def _load_ngd_csv(path: str) -> pd.DataFrame:
    """Load one OS NGD CSV and normalise column names based on product type."""
    fname = os.path.basename(path)
    if "_altadd" in fname:
        df = pd.read_csv(path, usecols=_ALT_COLS)
        df = df.rename(columns={
            "alternatesubname": "subname",
            "alternatename":    "name",
            "alternatenumber":  "number",
            "addressstatus":    "source",
        })
        df["source"] = "alternative"
    elif "_pstladd" in fname:
        df = pd.read_csv(path, usecols=_PSTL_COLS)
        df = df.rename(columns={
            "subbuildingname":   "subname",
            "buildingname":      "name",
            "buildingnumber":    "number",
            "thoroughfare":      "streetname",
            "dependentlocality": "locality",
            "posttown":          "townname",
        })
        df["source"] = "postal"
        df["number"] = pd.array(df["number"], dtype="Int64")
    else:
        df = pd.read_csv(path, usecols=_DEFAULT_COLS)
        df["source"] = "main"
    df["number_str"] = _number_str(df["number"])
    df["type"] = fname.split("_")[2].split(".")[0]
    return df


def _join(*parts) -> str:
    """Join non-NaN address parts with ', '."""
    return ", ".join(
        str(p).upper() for p in parts if pd.notna(p) and str(p) not in ("", "nan", "NAN")
    )


def build_canonical_key(
    df: pd.DataFrame,
    fields: list[str],
    uid_prefix: str = "c",
) -> pd.DataFrame:
    """
    Build a single-column canonical address key DataFrame from the OS dataframe.

    Parameters
    ----------
    df      : OS NGD dataframe (output of `load_os_df`)
    fields  : column names to concatenate, in order
    uid_prefix : prefix for the uid column name

    Returns
    -------
    DataFrame with columns [f'uid_{uid_prefix}', 'uprn', 'address_c']
    """
    subset = df[["uprn"] + fields].copy()
    subset["address_c"] = [
        _join(*row) for row in subset[fields].itertuples(index=False)
    ]
    result = subset[["uprn", "address_c"]].drop_duplicates().sort_values("uprn")
    result = result.reset_index(drop=True).reset_index().rename(
        columns={"index": f"uid_{uid_prefix}"}
    )
    result[f"uid_{uid_prefix}"] = result[f"uid_{uid_prefix}"].astype(str)
    return result


def build_messy_key(
    df: pd.DataFrame,
    fields: list[str],
    uid_prefix: str = "m",
) -> pd.DataFrame:
    """
    Build a single-column messy address key DataFrame from the LDC dataframe.

    Parameters
    ----------
    df      : LDC dataframe (output of `load_ldc_df`)
    fields  : column names to concatenate, in order
    uid_prefix : prefix for the uid column name

    Returns
    -------
    DataFrame with columns [f'uid_{uid_prefix}', 'uprn', 'premises_id', 'address_m']
    """
    subset = df[["uprn", "premises_id"] + fields].copy()
    subset["address_m"] = [
        _join(*row) for row in subset[fields].itertuples(index=False)
    ]
    result = subset[["uprn", "premises_id", "address_m"]].drop_duplicates().sort_values("premises_id")
    result = result.reset_index(drop=True).reset_index().rename(
        columns={"index": f"uid_{uid_prefix}"}
    )
    result[f"uid_{uid_prefix}"] = result[f"uid_{uid_prefix}"].astype(str)
    return result


def fuzzy_similarity(row) -> int:
    """fuzz.ratio between address_m and address_c columns."""
    s1 = str(row["address_m"]) if pd.notna(row["address_m"]) else ""
    s2 = str(row["address_c"]) if pd.notna(row["address_c"]) else ""
    return fuzz.ratio(s1, s2)


print("Helper functions defined.")

## 3. Load and clean OS NGD reference data

The OS NGD download comes as ZIP files, each containing multiple CSVs for
different address product types (`_altadd`, `_pstladd`, main). We:
1. Extract any ZIPs not already extracted
2. Load all CSVs into a single pandas DataFrame
3. Forward-fill classification and coordinates across address variants
4. Drop residential addresses (we only match commercial premises)
5. Add `postcode_sector` = postcode minus the last 2 characters

In [ ]:
# Extract ZIPs (skips files already extracted)
zip_list = [
    f for f in os.listdir(NGD_DATA_DIR)
    if f.endswith(".zip") and "streetaddress" not in f
]
for zf in zip_list:
    zip_path = os.path.join(NGD_DATA_DIR, zf)
    with zipfile.ZipFile(zip_path) as z:
        to_extract = [
            m for m in z.namelist()
            if "rltenty.csv" not in m and "otrclass.csv" not in m
        ]
        for member in to_extract:
            dest = os.path.join(NGD_DATA_DIR, member)
            if not os.path.exists(dest):
                z.extract(member, NGD_DATA_DIR)

print(f"ZIPs processed: {len(zip_list)}")

In [ ]:
# Load all CSVs
csv_files = [
    os.path.join(NGD_DATA_DIR, f)
    for f in os.listdir(NGD_DATA_DIR)
    if f.endswith(".csv")
]
print(f"Loading {len(csv_files)} CSV files...")

os_df = pd.concat([_load_ngd_csv(f) for f in csv_files], axis=0)
os_df = os_df.replace(pd.NA, np.nan)
print(f"Raw rows: {len(os_df):,}")

In [ ]:
# Forward-fill classification and coordinates across address variants,
# then drop residentials
os_df = os_df.sort_values(["uprn", "primaryclassificationdescription"])
os_df["primaryclassificationdescription"] = (
    os_df.groupby("uprn", sort=False)["primaryclassificationdescription"].ffill()
)
os_df = os_df[os_df["primaryclassificationdescription"] != "Residential"]
os_df["latitude"]  = os_df.groupby("uprn", sort=False)["latitude"].ffill()
os_df["longitude"] = os_df.groupby("uprn", sort=False)["longitude"].ffill()

# Deduplicate on key fields
os_df = os_df.drop_duplicates(
    subset=["uprn", "organisationname", "subname", "name",
            "number_str", "streetname", "townname", "postcode"],
    keep="first",
)

# Postcode sector = postcode without the last 2 characters  e.g. 'SW1W 0LN' -> 'SW1W 0'
os_df["postcode_sector"] = os_df["postcode"].str[:-2]

print(f"Commercial rows after cleaning: {len(os_df):,}")
os_df[["uprn", "organisationname", "name", "number", "streetname", "postcode"]].head(3)

## 4. Build canonical address keys (Methods A–F)

Each method encodes a different combination of OS address components into a single
string. Using multiple encodings increases the chance that at least one variant
matches the LDC input:

- **A** — org name + street + postcode sector *(best for named organisations)*
- **B** — sub-building + number + street + postcode sector
- **C** — building name + number + street + postcode sector
- **D** — building name + sub-building + street + postcode sector
- **E** — number + street + postcode sector *(most minimal, highest recall)*
- **F** — building name + sub-building + number + street + postcode sector

In [ ]:
canonical_a = build_canonical_key(os_df, ["organisationname", "streetname", "postcode_sector"], "ca")
canonical_b = build_canonical_key(os_df, ["subname",          "number_str",  "streetname", "postcode_sector"], "cb")
canonical_c = build_canonical_key(os_df, ["name",             "number_str",  "streetname", "postcode_sector"], "cc")
canonical_d = build_canonical_key(os_df, ["name",             "subname",     "streetname", "postcode_sector"], "cd")
canonical_e = build_canonical_key(os_df, ["number_str",       "streetname",  "postcode_sector"],               "ce")
canonical_f = build_canonical_key(os_df, ["name",     "subname", "number_str", "streetname", "postcode_sector"], "cf")

for label, df in [("A", canonical_a), ("B", canonical_b), ("C", canonical_c),
                  ("D", canonical_d), ("E", canonical_e), ("F", canonical_f)]:
    print(f"Canonical {label}: {len(df):,} rows — e.g. {df['address_c'].iloc[0]!r}")

## 5. Load LDC messy data

The LDC CSV has a single concatenated `address` field and an `organisationname`
(`tenant`). We split the address on `, ` (right-aligning so postcode is always
the last token), then drop the post town and city fields that OS doesn't match on.

In [ ]:
ldc_raw = pd.read_csv(LDC_FILE)
ldc_raw["address"] = (
    ldc_raw["address"]
    .str.replace("'", "", regex=False)
    .str.replace(".", "", regex=False)
)
print(f"LDC rows: {len(ldc_raw):,}")
ldc_raw[["premises_id", "tenant", "address", "uprn_id"]].head(3)

In [ ]:
# Split the concatenated address into columns, right-shift so postcode is last
ldc_split = ldc_raw["address"].str.split(", ", n=10, expand=True)
for _ in range(ldc_split.shape[1]):
    no_postcode = ldc_split.index[ldc_split[ldc_split.shape[1] - 1].isna()].tolist()
    ldc_split.iloc[no_postcode] = ldc_split.iloc[no_postcode].shift(periods=1, axis=1)

# Drop post-town (col -2) and city (col -3), add org name, UPRN, sector
n = ldc_split.shape[1]
ldc_df = ldc_split.drop(columns=[n - 3, n - 2])
ldc_df["organisationname"] = ldc_raw["tenant"]
ldc_df["uprn"]             = ldc_raw["uprn_id"]
ldc_df["premises_id"]      = ldc_raw["premises_id"]
ldc_df["postcode"]         = ldc_df[n - 1]
ldc_df["postcode_sector"]  = ldc_df["postcode"].str[:-2]
ldc_df["streetname"]       = ldc_df[n - 5]
ldc_df = ldc_df.drop_duplicates()

print(f"LDC after splitting: {len(ldc_df):,} rows")
ldc_df.head(2)

## 6. Build messy address keys (Methods a–b)

The LDC data doesn't have distinct OS-style fields (building name, sub-building etc.),
so we only need two messy variants:

- **a** — org name + street + postcode sector *(matches canonical A)*
- **b** — full raw address tokens + postcode sector *(matches canonicals B–F)*

In [ ]:
# Identify the token columns (everything between col 0 and streetname/postcode)
_addr_token_cols = [c for c in ldc_df.columns if isinstance(c, int) and c <= n - 5]

messy_a = build_messy_key(ldc_df, ["organisationname", "streetname", "postcode_sector"], "ma")
messy_b = build_messy_key(ldc_df, _addr_token_cols + ["streetname", "postcode_sector"], "mb")

print(f"Messy A: {len(messy_a):,} rows — e.g. {messy_a['address_m'].iloc[0]!r}")
print(f"Messy B: {len(messy_b):,} rows — e.g. {messy_b['address_m'].iloc[0]!r}")

## 7. Run 6 matching rounds

For each (messy, canonical) pair we:
1. Register the DataFrames as DuckDB relations
2. Run `AddressMatcher` (exact pass → Splink probabilistic pass)
3. Collect the raw match result
4. Join back to the original messy/canonical frames to recover `address_m`,
   `address_c`, `premises_id`, and `uprn`

The `final_match_weight_threshold=8` means Splink will only return pairs with
match weight ≥ 8 (~94% probability). Lower this value to trade precision for
recall.

In [ ]:
pair_list = [
    (messy_a, canonical_a, "a"),
    (messy_b, canonical_b, "b"),
    (messy_b, canonical_c, "c"),
    (messy_b, canonical_d, "d"),
    (messy_b, canonical_e, "e"),
    (messy_b, canonical_f, "f"),
]

_STAGES = [
    ExactMatchStage(),
    SplinkStage(
        predict_threshold_match_weight=-20,
        final_match_weight_threshold=8,
        include_full_postcode_block=False,
        retain_intermediate_calculation_columns=False,
    ),
]

all_results = []
t0 = time.time()

for i, (df_messy_pd, df_canon_pd, method) in enumerate(pair_list):
    uid_m_col = [c for c in df_messy_pd.columns if c.startswith("uid_")][0]
    uid_c_col = [c for c in df_canon_pd.columns if c.startswith("uid_")][0]

    # Register as DuckDB relations — AddressMatcher expects unique_id + address_concat
    df_m = con.from_df(
        df_messy_pd.rename(columns={uid_m_col: "unique_id", "address_m": "address_concat"})
        [["unique_id", "address_concat"]]
        .dropna(subset=["address_concat"])
    )
    df_c = con.from_df(
        df_canon_pd.rename(columns={uid_c_col: "unique_id", "address_c": "address_concat", "uprn": "uprn_c"})
        [["unique_id", "address_concat"]]
        .dropna(subset=["address_concat"])
    )

    print(f"Round {i} (method '{method}') — {df_messy_pd.shape[0]:,} messy × {df_canon_pd.shape[0]:,} canonical")
    t_round = time.time()

    matcher = AddressMatcher(
        canonical_addresses=df_c,
        addresses_to_match=df_m,
        con=con,
        stages=_STAGES,
    )
    match_result = matcher.match()
    df_raw = match_result.matches(all_columns=True).df()

    if df_raw.empty:
        print(f"  No matches found — skipping.")
        continue

    # Join back to recover address_m, address_c, premises_id, uprn
    df_raw = df_raw.rename(columns={"unique_id": uid_m_col})
    df_raw[uid_m_col] = df_raw[uid_m_col].astype(str)

    merged = df_raw.merge(
        df_messy_pd[[uid_m_col, "premises_id", "address_m", "uprn"]].rename(columns={"uprn": "uprn_m"}),
        on=uid_m_col,
        how="left",
    )
    merged = merged.merge(
        df_canon_pd[[uid_c_col, "address_c", "uprn"]].rename(
            columns={uid_c_col: "resolved_canonical_id", "uprn": "uprn_c"}
        ),
        on="resolved_canonical_id",
        how="left",
    )
    merged["method"] = method
    all_results.append(merged)
    print(f"  → {len(merged):,} matches in {time.time() - t_round:.1f}s")

print(f"\nAll rounds done in {time.time() - t0:.1f}s")

## 8. Combine results and compute fuzzy similarity

`fuzz_similarity` is the Levenshtein ratio between the LDC string we sent into
the matcher (`address_m`) and the OS string it matched against (`address_c`).
A score of 100 means the strings were identical; ≥ 95 is near-identical.

In [ ]:
full_results = pd.concat(all_results, ignore_index=True)

full_results["fuzz_similarity"] = full_results.apply(fuzzy_similarity, axis=1)

print(f"Total match rows  : {len(full_results):,}")
print(f"Unique premises   : {full_results['premises_id'].nunique():,}")
print(f"fuzz_similarity distribution:")
print(full_results["fuzz_similarity"].describe().to_string())

## 9. Apply hit rules

A **hit** is a match we are confident enough to accept.  Three independent rules
each contribute a set of `premises_id` values:

| Rule | Condition | Rationale |
|------|-----------|-----------|
| **p1** | `fuzz_similarity == 100` | Exact string match — zero ambiguity |
| **p2** | `fuzz_similarity > 95` **and** `method == 'a'` | Org-name round, near-exact |
| **p3** | `distinguishability IS NOT NULL` | Splink found a unique token |

For each rule, when a `premises_id` appears in multiple rows (different methods
or match rounds) we take the **highest `match_weight`** row as the accepted match.

In [ ]:
def top_by_weight(df: pd.DataFrame) -> pd.Series:
    """Return series of premises_id values that have at least one qualifying row."""
    return (
        df.sort_values("match_weight", ascending=False)
        .groupby("premises_id")
        .first()
        .reset_index()["premises_id"]
    )


p1 = top_by_weight(full_results[full_results["fuzz_similarity"] == 100])
p2 = top_by_weight(full_results[(full_results["fuzz_similarity"] > 95) & (full_results["method"] == "a")])
p3 = top_by_weight(full_results[full_results["distinguishability"].notna()])

hit_premises = set(p1) | set(p2) | set(p3)
all_premises = set(full_results["premises_id"].dropna())

coverage = len(hit_premises) / len(all_premises) if all_premises else 0

print(f"Rule p1 (exact string match)     : {len(p1):,} premises")
print(f"Rule p2 (org near-exact, fuzz>95): {len(p2):,} premises")
print(f"Rule p3 (distinguishability)     : {len(p3):,} premises")
print(f"Union (any rule)                 : {len(hit_premises):,} / {len(all_premises):,}")
print(f"\nCoverage: {coverage:.1%}")

## 10. Explore results

The cells below let you inspect which premises were and were not matched,
and dig into edge cases.

In [ ]:
# Premises NOT matched by any rule — review these to find new hit criteria
unmatched = full_results[~full_results["premises_id"].isin(hit_premises)]
print(f"Unmatched premises: {unmatched['premises_id'].nunique():,}")
unmatched[["premises_id", "address_m", "address_c", "match_weight", "fuzz_similarity", "method"]].head(10)

In [ ]:
# Coverage breakdown by method
full_results.groupby("method").agg(
    rows=("premises_id", "count"),
    premises=("premises_id", "nunique"),
    avg_fuzz=("fuzz_similarity", "mean"),
    avg_weight=("match_weight", "mean"),
).round(1)

In [ ]:
# Best accepted match for each premises (one row per premises_id)
best_matches = (
    full_results[full_results["premises_id"].isin(hit_premises)]
    .sort_values("match_weight", ascending=False)
    .groupby("premises_id")
    .first()
    .reset_index()
)
print(f"Best-match rows: {len(best_matches):,}")
best_matches[["premises_id", "uprn_c", "address_m", "address_c",
              "match_weight", "fuzz_similarity", "method"]].head(10)